# Week 2 — Data Preparation
**Project:** Drug Side Effect Predictor  
**Goal:** Merge SIDER files, clean attributes, filter labels, build the modeling table, and split into train/test sets.

**Decisions locked in from Week 1:**
- Drop drugs with missing ATC codes (312 drugs dropped, 1,118 remain)
- Label frequency threshold: ≥25 (keeps 919 side effects out of 4,251)
- Train/test split: 80/20 at the drug level

## Setup

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Hardcoded paths
RAW       = r"D:\Experimental Stuff\Data Science\Drug-Side-Effect-Predictor\data\raw\\"
PROCESSED = r"D:\Experimental Stuff\Data Science\Drug-Side-Effect-Predictor\data\processed\\"
SPLITS    = r"D:\Experimental Stuff\Data Science\Drug-Side-Effect-Predictor\data\splits\\"

print("Libraries loaded.")

Libraries loaded.


---
## Step 2.1 — Merge SIDER files into one drug table

We load the same five files from Week 1 and join them on `stitch_id_flat`.

**Join logic:**
- Start with `drug_names` as the base (1,430 drugs)
- Left join `drug_atc` — drugs without an ATC code get NaN
- Left join aggregated `indications` — drugs without an indication get NaN
- Keep only drugs that have at least one side effect label

We are NOT dropping missing values here yet — that happens in Step 2.2.

In [4]:
# Load all five files — same as Week 1
drug_names = pd.read_csv(
    RAW + "drug_names.tsv",
    sep="\t", header=None,
    names=["stitch_id_flat", "drug_name"]
)

drug_atc = pd.read_csv(
    RAW + "drug_atc.tsv",
    sep="\t", header=None,
    names=["stitch_id_flat", "atc_code"]
)

meddra = pd.read_csv(
    RAW + "meddra.tsv",
    sep="\t", header=None,
    names=["umls_id", "meddra_id", "term_type", "side_effect_name"]
)

se = pd.read_csv(
    RAW + "meddra_all_se.tsv",
    sep="\t", header=None,
    names=["stitch_id_flat", "stitch_id_stereo", "umls_id_label",
           "meddra_type", "umls_id_meddra", "side_effect_name"]
)

indications = pd.read_csv(
    RAW + "meddra_all_indications.tsv",
    sep="\t", header=None,
    names=["stitch_id_flat", "umls_id_label", "method",
           "concept_name", "meddra_type", "umls_id_meddra", "meddra_concept_name"]
)

print("All files loaded.")

All files loaded.


In [5]:
# Filter PT only for side effects and indications (same reasoning as Week 1)
se_pt  = se[se["meddra_type"] == "PT"].copy()
ind_pt = indications[indications["meddra_type"] == "PT"].copy()

print("Side effect rows (PT only):", len(se_pt))
print("Indication rows (PT only): ", len(ind_pt))

Side effect rows (PT only): 163206
Indication rows (PT only):  16204


In [6]:
# Aggregate ATC codes per drug — a drug can have multiple ATC codes
# We join them with a pipe separator: e.g. "A02BC01|N02BE01"
atc_agg = (
    drug_atc
    .groupby("stitch_id_flat")["atc_code"]
    .apply(lambda x: "|".join(x))
    .reset_index()
)
atc_agg.columns = ["stitch_id_flat", "atc_codes"]

# Aggregate indications per drug into a single text string (for TF-IDF later)
ind_agg = (
    ind_pt
    .groupby("stitch_id_flat")["meddra_concept_name"]
    .apply(lambda x: " ".join(x))
    .reset_index()
)
ind_agg.columns = ["stitch_id_flat", "indications_text"]

print("Drugs with ATC codes:", len(atc_agg))
print("Drugs with indications:", len(ind_agg))

Drugs with ATC codes: 1118
Drugs with indications: 1437


In [7]:
# Build the merged drug table
# Start with drug_names, left join ATC and indications
drug_table = drug_names.merge(atc_agg, on="stitch_id_flat", how="left")
drug_table = drug_table.merge(ind_agg, on="stitch_id_flat", how="left")

# Keep only drugs that appear in the side effect file
drugs_with_labels = set(se_pt["stitch_id_flat"])
drug_table = drug_table[drug_table["stitch_id_flat"].isin(drugs_with_labels)].copy()

print("Merged drug table shape:", drug_table.shape)
print()
print("Missing ATC codes:    ", drug_table["atc_codes"].isna().sum())
print("Missing indications:  ", drug_table["indications_text"].isna().sum())
print()

# Check for join mismatches — no drug should appear more than once
duplicates = drug_table["stitch_id_flat"].duplicated().sum()
print("Duplicate drug IDs (should be 0):", duplicates)
print()
drug_table.head()

Merged drug table shape: (1430, 4)

Missing ATC codes:     312
Missing indications:   70

Duplicate drug IDs (should be 0): 0



,stitch_id_flat,drug_name,atc_codes,indications_text
0,CID100000085,carnitine,A16AA01,Failure to thrive Hypoglycaemia Renal failure ...
1,CID100000119,gamma-aminobutyric,L03AA03|N03AG03,Hypogammaglobulinaemia Varicella Immunodeficie...
2,CID100000137,5-aminolevulinic,L01XD04,Actinic keratosis Seborrhoeic keratosis Porphy...
3,CID100000143,leucovorin,V03AF03|V03AF04|V03AF06,Anaemia Anaemia megaloblastic Pernicious anaem...
4,CID100000146,5-methyltetrahydrofolate,NaN,NaN


---
## Step 2.2 — Clean drug-level attributes

**Decisions:**
- Drop drugs with missing ATC codes (312 drugs) — we cannot impute therapeutic class
- Fill missing indications with empty string `""` — these drugs simply have no known indication text; TF-IDF will produce an all-zero vector for them, which is honest
- Standardise string formatting: lowercase and strip whitespace on drug names and indications

In [8]:
# DECISION: Drop drugs with no ATC code
# Reason: ATC code is our main categorical feature (therapeutic class).
# Imputing an unknown therapeutic class would introduce false signal.
before = len(drug_table)
drug_table = drug_table[drug_table["atc_codes"].notna()].copy()
after = len(drug_table)

print(f"Drugs before dropping missing ATC: {before}")
print(f"Drugs after dropping missing ATC:  {after}")
print(f"Drugs dropped: {before - after}")

Drugs before dropping missing ATC: 1430
Drugs after dropping missing ATC:  1118
Drugs dropped: 312


In [9]:
# DECISION: Fill missing indications with empty string
# Reason: 70 drugs have no indication text. An empty string produces
# an all-zero TF-IDF vector — the model simply learns nothing from
# the indication feature for these drugs, which is correct.
drug_table["indications_text"] = drug_table["indications_text"].fillna("")

print("Missing indications after fill:", drug_table["indications_text"].isna().sum())

Missing indications after fill: 0


In [10]:
# Standardise string formatting
# Lowercase and strip whitespace on drug_name and indications_text
drug_table["drug_name"]        = drug_table["drug_name"].str.lower().str.strip()
drug_table["indications_text"] = drug_table["indications_text"].str.lower().str.strip()

# ATC codes: uppercase is the standard format — keep as-is, just strip whitespace
drug_table["atc_codes"] = drug_table["atc_codes"].str.strip()

print("String formatting done.")
print()
print("Final drug table shape:", drug_table.shape)
print()
drug_table.head()

String formatting done.

Final drug table shape: (1118, 4)



,stitch_id_flat,drug_name,atc_codes,indications_text
0,CID100000085,carnitine,A16AA01,failure to thrive hypoglycaemia renal failure ...
1,CID100000119,gamma-aminobutyric,L03AA03|N03AG03,hypogammaglobulinaemia varicella immunodeficie...
2,CID100000137,5-aminolevulinic,L01XD04,actinic keratosis seborrhoeic keratosis porphy...
3,CID100000143,leucovorin,V03AF03|V03AF04|V03AF06,anaemia anaemia megaloblastic pernicious anaem...
5,CID100000158,pge2,G02AD02,abdominal pain abortion missed cardiovascular ...


---
## Step 2.3 — Filter the label space

**Decision:** Keep only side effects that appear in ≥25 drugs.

**Reason:** Side effects with fewer than 25 examples are too rare for the model to learn a reliable pattern. Including them would add noise and hurt overall performance.

We also restrict the side effect file to only the 1,118 drugs remaining after the ATC drop — otherwise we would be counting associations from drugs we have already removed.

In [11]:
# Restrict side effects to drugs remaining in our cleaned table
remaining_drugs = set(drug_table["stitch_id_flat"])
se_filtered = se_pt[se_pt["stitch_id_flat"].isin(remaining_drugs)].copy()

print("Side effect rows before drug filter:", len(se_pt))
print("Side effect rows after drug filter: ", len(se_filtered))

Side effect rows before drug filter: 163206
Side effect rows after drug filter:  141209


In [12]:
# Count how many drugs each side effect appears in
se_counts = se_filtered["side_effect_name"].value_counts()

print("Label frequency distribution (after ATC drop):")
for threshold in [1, 5, 10, 25, 50, 100]:
    count = (se_counts >= threshold).sum()
    print(f"  Side effects in >= {threshold:3d} drugs: {count}")

Label frequency distribution (after ATC drop):
  Side effects in >=   1 drugs: 4073
  Side effects in >=   5 drugs: 1937
  Side effects in >=  10 drugs: 1369
  Side effects in >=  25 drugs: 845
  Side effects in >=  50 drugs: 549
  Side effects in >= 100 drugs: 350


In [13]:
# DECISION: Apply threshold of 25
LABEL_THRESHOLD = 25

valid_labels = se_counts[se_counts >= LABEL_THRESHOLD].index.tolist()

print(f"Side effects kept (>= {LABEL_THRESHOLD} drugs): {len(valid_labels)}")
print(f"Side effects dropped:                        {len(se_counts) - len(valid_labels)}")

# Filter the association file to valid labels only
se_final = se_filtered[se_filtered["side_effect_name"].isin(valid_labels)].copy()

print()
print("Association rows after label filter:", len(se_final))

Side effects kept (>= 25 drugs): 845
Side effects dropped:                        3228

Association rows after label filter: 125475


---
## Step 2.4 — Build the drug-level modeling table

We pivot the side effect associations into binary columns.

**Structure of the final table:**
- One row per drug (~1,118 rows)
- Columns: `stitch_id_flat`, `drug_name`, `atc_codes`, `indications_text`, then one binary column per side effect (~919 columns)
- Value 1 = this drug has a known association with this side effect
- Value 0 = not in the dataset (no known association)

**Important note on zeros:** A 0 does not mean the drug definitely does NOT cause that side effect — it means it was not recorded in SIDER. This is the open-world assumption and is standard in multi-label drug datasets.

In [14]:
# Pivot: one row per drug, one column per side effect, value = 1
se_final["value"] = 1

label_matrix = se_final.pivot_table(
    index="stitch_id_flat",
    columns="side_effect_name",
    values="value",
    aggfunc="max"        # if a drug-SE pair appears twice, keep 1
).fillna(0).astype(int)  # fill missing associations with 0

print("Label matrix shape:", label_matrix.shape)
print("(should be ~1118 drugs x ~919 side effects)")
print()
label_matrix.head()

Label matrix shape: (1117, 845)
(should be ~1118 drugs x ~919 side effects)



side_effect_name,Abdominal discomfort,Abdominal distension,Abdominal pain,Abdominal pain lower,Abdominal pain upper,Abnormal behaviour,Abnormal dreams,Abnormal faeces,Abscess,Accommodation disorder,...,Vomiting,Vulvovaginal candidiasis,Vulvovaginal mycotic infection,Weight decreased,Weight increased,Wheezing,White blood cell count decreased,Withdrawal syndrome,Xerophthalmia,Yawning
stitch_id_flat,,,,,,,,,,,,,,,,,,,,,
CID100000085,0,0,1,0,0,0,0,0,0,0,...,1,0,0,1,1,0,0,0,0,0
CID100000119,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
CID100000137,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
CID100000143,0,0,1,1,1,0,0,0,0,0,...,1,0,0,0,0,1,0,0,0,0
CID100000158,0,0,1,0,0,0,0,0,0,0,...,1,0,0,0,0,1,0,0,0,0


In [15]:
# Merge drug attributes with label matrix
modeling_table = drug_table.merge(
    label_matrix.reset_index(),
    on="stitch_id_flat",
    how="inner"  # inner join: keep only drugs that survived both the ATC drop and label filter
)

print("Final modeling table shape:", modeling_table.shape)
print("(rows = drugs, columns = 4 attribute cols + ~919 label cols)")
print()

# Sanity check: no drug should appear twice
print("Duplicate drug IDs (should be 0):", modeling_table["stitch_id_flat"].duplicated().sum())
print()

# Check overall label density (what % of drug-label pairs are 1)
label_cols = valid_labels
density = modeling_table[label_cols].values.mean() * 100
print(f"Label density: {density:.2f}% of drug-label pairs are positive")
print("(Low density is expected — most drugs do not have most side effects)")

Final modeling table shape: (1117, 849)
(rows = drugs, columns = 4 attribute cols + ~919 label cols)

Duplicate drug IDs (should be 0): 0

Label density: 11.61% of drug-label pairs are positive
(Low density is expected — most drugs do not have most side effects)


In [16]:
# Save the full modeling table to processed/
output_path = PROCESSED + "modeling_table.csv"
modeling_table.to_csv(output_path, index=False)

print(f"Modeling table saved to: {output_path}")
print(f"Shape: {modeling_table.shape}")

Modeling table saved to: D:\Experimental Stuff\Data Science\Drug-Side-Effect-Predictor\data\processed\\modeling_table.csv
Shape: (1117, 849)


---
## Step 2.5 — Drug-level train/test split

**Why drug-level split matters:**  
We split on the drug, not on individual drug-side effect pairs. If we split on pairs, the same drug could appear in both train and test — the model would have seen it during training and would perform artificially well on test. This is called **data leakage**.

**Split:** 80% train, 20% test  
**Random state:** 42 (fixed for reproducibility — anyone running this notebook gets the same split)

In [17]:
# Split at the drug level
train_df, test_df = train_test_split(
    modeling_table,
    test_size=0.2,
    random_state=42
)

print("Train set shape:", train_df.shape)
print("Test set shape: ", test_df.shape)
print()

# Verify no drug appears in both sets
train_ids = set(train_df["stitch_id_flat"])
test_ids  = set(test_df["stitch_id_flat"])
overlap   = train_ids & test_ids

print(f"Drug IDs in train:     {len(train_ids)}")
print(f"Drug IDs in test:      {len(test_ids)}")
print(f"Overlap (should be 0): {len(overlap)}")

Train set shape: (893, 849)
Test set shape:  (224, 849)

Drug IDs in train:     893
Drug IDs in test:      224
Overlap (should be 0): 0


In [18]:
# Save train and test splits
train_path = SPLITS + "train.csv"
test_path  = SPLITS + "test.csv"

train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path,  index=False)

print(f"Train set saved to: {train_path}")
print(f"Test set saved to:  {test_path}")

Train set saved to: D:\Experimental Stuff\Data Science\Drug-Side-Effect-Predictor\data\splits\\train.csv
Test set saved to:  D:\Experimental Stuff\Data Science\Drug-Side-Effect-Predictor\data\splits\\test.csv


---
## Week 2 Summary

| Step | Decision | Result |
|---|---|---|
| 2.1 Merge files | Joined on `stitch_id_flat` | 1,430 drugs with labels |
| 2.2 Clean attributes | Dropped 312 drugs with no ATC code; filled 70 missing indications with `""` | 1,118 drugs remain |
| 2.3 Filter labels | Kept side effects appearing in >=25 drugs | ~919 labels kept |
| 2.4 Modeling table | One row per drug, binary label columns | ~1,118 x ~923 table |
| 2.5 Train/test split | 80/20 split at drug level, random_state=42 | ~894 train / ~224 test, zero leakage |

**Files saved:**
- `data/processed/modeling_table.csv` — full cleaned dataset
- `data/splits/train.csv` — training set
- `data/splits/test.csv` — test set

**Next: Week 3 — Feature Engineering**  
Extract TF-IDF features from `indications_text`, encode `atc_codes`, and prepare the feature matrix X and label matrix y for modeling.